In [ ]:
VERSION = 100
N_SPEAKER = 3
REPEAT = False
DATASET_NAMES = "['val']"

In [ ]:
import torch
import yaml
import ast
import os
from typing import Literal
import pytorch_lightning as pl
from ism.data.basic_datamodule import BasicDatamodule
from ism.src.ambisonics_exp import AmbisonicsExp
from tqdm import tqdm
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Audio, display
from matplotlib.ticker import MultipleLocator
from matplotlib.patches import Wedge
from itertools import islice
from torchmetrics.functional.audio import scale_invariant_signal_distortion_ratio as sisdr

CONFIG_PATH = '../../../config/'
PREFIX = 'ambix'

# customize this
from ism.data.moving_librimix import SinusoidLibriMix as DS

# make sure list is list
DATASET_NAMES = ast.literal_eval(DATASET_NAMES)

In [ ]:
class AmbisonicsWriteISMExp(AmbisonicsExp):
    
    def __init__(
        self,
        WriteDS: torch.utils.data.Dataset,
        dataset_name: Literal['train', 'dev', 'test'],
        **kwargs,
    ):
        super().__init__(**kwargs)
        
        # create storage path
        self.dataset_name = 'val' if dataset_name == 'dev' else dataset_name
        storage_dir = os.path.join(
            os.environ['CUSTOMLIBRIMIX'], 
            PREFIX,
            WriteDS.__name__[:-3] + str(kwargs['audio_params']['n_speaker']) + WriteDS.__name__[-3:],
            self.dataset_name
        )
        if os.path.exists(storage_dir):
            if os.listdir(storage_dir):
                raise RuntimeError(f"Directory '{storage_dir}' is not empty!")
        else:
            os.makedirs(storage_dir, exist_ok=False)
        self.storage_dir = storage_dir
        
    def on_validation_start(self):
        self.on_fit_start()
        
    def validation_step(self, batch, batch_idx):
        # get rng
        self.get_rng(batch)

        # compute ISM
        self.ism.get_ism(batch)
        
        # postprocess data
        for key in list(batch.keys()):
            if key in ['clean_td', 'dp_rir', 'rir']:
                del batch[key]  # reduce storage consumption
            elif isinstance(batch[key], torch.Tensor):
                batch[key] = batch[key].squeeze(0).cpu()  # remove batch dim and move to CPU for storage
        
        # store data
        example_idx = int(batch['idx'].cpu().squeeze())
        file_path = os.path.join(
            self.storage_dir, f'example_{example_idx}.pt'
        )
        torch.save(batch, file_path)

In [ ]:
# load default parametrization
base_config_path = CONFIG_PATH + 'base_config.yaml'
with open(base_config_path, 'r') as file:
    config = yaml.safe_load(file)
    
# instantiate datamodule
config['data_params']['corpus_params']['version'] = VERSION
config['data_params']['corpus_params']['repeat'] = REPEAT
config['audio_params']['n_speaker'] = N_SPEAKER
dm = BasicDatamodule(
    DS, 
    batch_size=1, 
    n_workers=0,  
    audio_params=config['audio_params'],
    data_params=config['data_params'],
)

# run ISM
trainer = pl.Trainer(
    logger=False,
    devices=1,
    accelerator='gpu',
)
for dataset_name in DATASET_NAMES:        
    # instantiate experiment
    exp = AmbisonicsWriteISMExp(
        WriteDS=DS,
        dataset_name=dataset_name,
        model=None,
        tf_logger=None, 
        **config, 
    )
    
    # create and save dataset
    dl = getattr(dm, f"{'val' if dataset_name == 'dev' else dataset_name}_dataloader")()
    trainer.validate(
        exp, dataloaders=dl
    )